In [104]:
import pandas as pd
import numpy as np
from google.colab import files
from google.colab import drive
import os

# Hubungkan Google Drive
drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/UAS_ETL'))
# Folder output ETL
folder_path = "/content/drive/MyDrive/UAS_ETL"
os.makedirs(folder_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['hasil_etl_dbd_airminum.csv', 'Data_DBD_2023.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2023.csv', 'Data_DBD_2021.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2024.csv', 'Data_DBD_2022.csv', 'Data_DBD_2024.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2022.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2021.csv']


In [105]:
# ==================================================
# 2. EXTRACT DATA
# Membaca dataset dari Google Drive
# ==================================================

dbd21 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2021.csv')
dbd22 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2022.csv')
dbd23 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2023.csv')
dbd24 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2024.csv')

air21 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2021.csv')
air22 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2022.csv')
air23 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2023.csv')
air24 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2024.csv')

In [106]:
# ==================================================
# Membaca data
# ==================================================
print("DBD 2021 :", dbd21.shape)
print("DBD 2022 :", dbd22.shape)
print("DBD 2023 :", dbd23.shape)
print("DBD 2024 :", dbd24.shape)

print("AIR 2021 :", air21.shape)
print("AIR 2022 :", air22.shape)
print("AIR 2023 :", air23.shape)
print("AIR 2024 :", air24.shape)

DBD 2021 : (35, 7)
DBD 2022 : (35, 7)
DBD 2023 : (39, 7)
DBD 2024 : (39, 7)
AIR 2021 : (42, 4)
AIR 2022 : (42, 4)
AIR 2023 : (42, 4)
AIR 2024 : (42, 4)


In [107]:
# ==================================================
# 3. ADD YEAR
# ==================================================

dbd21['Tahun'] = 2021
dbd22['Tahun'] = 2022
dbd23['Tahun'] = 2023
dbd24['Tahun'] = 2024

air21['Tahun'] = 2021
air22['Tahun'] = 2022
air23['Tahun'] = 2023
air24['Tahun'] = 2024

In [108]:
# ==================================================
# 4. CONCAT DATA
# Menggabungkan data seluruh tahun
# ==================================================

dbd = pd.concat(
    [dbd21, dbd22, dbd23, dbd24],
    ignore_index=True
)

air = pd.concat(
    [air21, air22, air23, air24],
    ignore_index=True
)

In [109]:
# ==================================================
# 5. DATA CLEANING
# Menghapus missing value dan data duplikat
# ==================================================

def clean_dataframe(df):

    df = df.dropna()

    df = df.drop_duplicates()

    return df

In [110]:
dbd = clean_dataframe(dbd)
air = clean_dataframe(air)

In [111]:
# ==================================================
# 6. STANDARDIZE DATA
# Menyeragamkan format nama provinsi
# ==================================================

dbd['Provinsi'] = (
    dbd['Provinsi']
    .str.upper()
    .str.strip()
)

air['38 Provinsi'] = (
    air['38 Provinsi']
    .str.upper()
    .str.strip()
)

In [112]:
# ==================================================
# 7. SELECT COLUMN
# Memilih kolom yang dibutuhkan
# ==================================================

dbd = dbd[
    ['Provinsi','Tahun','Jumlah Kasus']
]

air = air[
    ['38 Provinsi','Tahun','Unnamed: 3']
]

In [115]:
# ==================================================
# 8. DATA INTEGRATION
# Menggabungkan data DBD dan Air Minum
# ==================================================

# 1. Hapus agregat nasional
air = air[air['38 Provinsi'] != 'INDONESIA']

# 2. Samakan nama provinsi
air['38 Provinsi'] = air['38 Provinsi'].replace({
    'KEP. BANGKA BELITUNG': 'KEPULAUAN BANGKA BELITUNG',
    'KEP. RIAU': 'KEPULAUAN RIAU'
})

# 3. Cek kesesuaian provinsi
provinsi_dbd = set(dbd['Provinsi'].unique())
provinsi_air = set(air['38 Provinsi'].unique())

hilang = provinsi_dbd - provinsi_air

print("Provinsi yang hilang:")
print(hilang)

# 4. Merge data
data_final = pd.merge(
    dbd,
    air,
    left_on=['Provinsi','Tahun'],
    right_on=['38 Provinsi','Tahun'],
    how='inner'
)

Provinsi yang hilang:
{'INDONESIA'}


In [116]:
# ==================================================
# 9. DATA QUALITY REPORT
# ==================================================

print("Jumlah Data :", len(data_final))

print(
    "Jumlah Provinsi :",
    data_final['Provinsi'].nunique()
)

print(
    "Rentang Tahun :",
    data_final['Tahun'].min(),
    "-",
    data_final['Tahun'].max()
)

print(
    "Missing Value :",
    data_final.isnull().sum().sum()
)

print(
    "Duplicate Data :",
    data_final.duplicated().sum()
)

Jumlah Data : 144
Jumlah Provinsi : 38
Rentang Tahun : 2021 - 2024
Missing Value : 0
Duplicate Data : 0


In [118]:
# ==================================================
# 10. FINAL TRANSFORMATION
# Membersihkan dan mengganti nama kolom
# ==================================================

# The columns '38 Provinsi' and 'Unnamed: 3' were already handled
# by previous cells (k1fMupPx8bOQ and 5SB6Quylzlo9).
# Therefore, the initial drop and rename operations are redundant here.

# Rename 'Unnamed: 3' to 'Akses_Air_Minum' before using it
data_final = data_final.rename(columns={'Unnamed: 3': 'Akses_Air_Minum'})

data_final['Akses_Air_Minum'] = pd.to_numeric(
    data_final['Akses_Air_Minum'],
    errors='coerce'
)

data_final = data_final.dropna()

print(data_final.info())
print(data_final.head())
print(data_final.shape)

print(data_final.isnull().sum())

print(data_final.info())

print(data_final['Provinsi'].nunique())

print(sorted(data_final['Tahun'].unique()))

print("=== DATA QUALITY REPORT ===")

print("Shape :", data_final.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 143
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Provinsi         140 non-null    object 
 1   Tahun            140 non-null    int64  
 2   Jumlah Kasus     140 non-null    int64  
 3   38 Provinsi      140 non-null    object 
 4   Akses_Air_Minum  140 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 6.6+ KB
None
         Provinsi  Tahun  Jumlah Kasus     38 Provinsi  Akses_Air_Minum
0            ACEH   2021           366            ACEH            88.79
1  SUMATERA UTARA   2021          2918  SUMATERA UTARA            90.89
2  SUMATERA BARAT   2021           654  SUMATERA BARAT            83.40
3            RIAU   2021          1038            RIAU            89.76
4           JAMBI   2021           357           JAMBI            79.70
(140, 5)
Provinsi           0
Tahun              0
Jumlah Kasus       0
38 Provi

In [153]:
# ==================================================
# 11. STANDARDIZE COLUMN NAME
# ==================================================

# Step 1: Rename 'Jumlah Kasus' to 'Jumlah_Kasus' for consistency
if 'Jumlah Kasus' in data_final.columns:
    data_final = data_final.rename(columns={'Jumlah Kasus': 'Jumlah_Kasus'})
else:
    print("Warning: 'Jumlah Kasus' column not found for renaming in 5SB6Quylzlo9. Check previous steps.")

# Step 2: Drop '38 Provinsi' as it's redundant after merge and not needed for final output
if '38 Provinsi' in data_final.columns:
    data_final = data_final.drop(columns=['38 Provinsi'])
else:
    print("Warning: '38 Provinsi' column not found for dropping in 5SB6Quylzlo9. Check previous steps.")

# Step 3: Explicitly select and order the columns to be passed to the next stage
# This ensures only desired columns are kept and in a specific order
final_columns_after_5SB6Quylzlo9 = ['Provinsi', 'Tahun', 'Jumlah_Kasus', 'Akses_Air_Minum']

# Verify all desired columns are present before selection
missing_cols_pre_filter = [col for col in final_columns_after_5SB6Quylzlo9 if col not in data_final.columns]
if missing_cols_pre_filter:
    print(f"Error: Columns {missing_cols_pre_filter} are missing in data_final before final selection in cell 5SB6Quylzlo9. Data_final will be empty.")
    data_final = pd.DataFrame(columns=final_columns_after_5SB6Quylzlo9) # Create an empty DataFrame to prevent downstream errors
else:
    data_final = data_final[final_columns_after_5SB6Quylzlo9]

print("Columns after 5SB6Quylzlo9:", data_final.columns)

Error: Columns ['Provinsi', 'Tahun', 'Jumlah_Kasus', 'Akses_Air_Minum'] are missing in data_final before final selection in cell 5SB6Quylzlo9. Data_final will be empty.
Columns after 5SB6Quylzlo9: Index(['Provinsi', 'Tahun', 'Jumlah_Kasus', 'Akses_Air_Minum'], dtype='object')


In [142]:
# ==================================================
# 12. EXPORT DATA
# Menyimpan hasil ETL ke CSV
# ==================================================

output_file = (
    '/content/drive/MyDrive/UAS_ETL/'
    'hasil_etl_dbd_airminum.csv'
)

data_final.to_csv(
    output_file,
    index=False
)
print("File berhasil disimpan ke Google Drive")


File berhasil disimpan ke Google Drive


In [143]:
# ==================================================
# 13. LOAD DATA TO DATABASE
# Memuat hasil ETL ke MySQL Aiven
# ==================================================
!pip install sqlalchemy psycopg2-binary pymysql

In [144]:
from sqlalchemy import create_engine

def connect_aiven():

    DATABASE_URL = (
        "mysql+pymysql://avnadmin:AVNS_c6K9EHiaWNQ2cyyu6we"
        "@mysql-4c516e3-data-engineering.d.aivencloud.com:19977/defaultdb"
    )

    engine = create_engine(
        DATABASE_URL,
        connect_args={
            "ssl": {
                "ssl_mode": "REQUIRED"
            }
        }
    )

    return engine

In [145]:
from sqlalchemy import text

def create_table(engine):

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS dbd_airminum (
                id INT PRIMARY KEY,
                Provinsi VARCHAR(100),
                Tahun INT,
                Jumlah_Kasus INT,
                Akses_Air_Minum FLOAT
            )
        """))

    print("Tabel berhasil dibuat")

In [146]:
if 'id' not in data_final.columns:
    data_final.insert(
        0,
        'id',
        range(1, len(data_final)+1)
    )

In [154]:
# ==================================================
# 12. FINAL COLUMN RENAMING AND ORDERING FOR DATABASE
# ==================================================

# Step 1: Rename columns to lowercase for the database
column_rename_map = {
    'Provinsi': 'provinsi',
    'Tahun': 'tahun',
    'Jumlah_Kasus': 'jumlah_kasus',
    'Akses_Air_Minum': 'akses_air_minum'
}
data_final = data_final.rename(columns=column_rename_map)

# Step 2: Explicitly select and order the columns, excluding 'id' which is added later
final_columns_for_db_pre_id = ['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum']

# Verify all desired columns are present before selection
missing_cols_pre_filter = [col for col in final_columns_for_db_pre_id if col not in data_final.columns]
if missing_cols_pre_filter:
    print(f"Error: Columns {missing_cols_pre_filter} are missing in data_final before final selection in cell F3IwRkvDrGTf. Data_final will be empty.")
    data_final = pd.DataFrame(columns=final_columns_for_db_pre_id) # Create an empty DataFrame to prevent downstream errors
else:
    data_final = data_final[final_columns_for_db_pre_id]

print("Columns after F3IwRkvDrGTf:", data_final.columns)
print("Data types after F3IwRkvDrGTf:")
print(data_final.dtypes)

Columns after F3IwRkvDrGTf: Index(['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum'], dtype='object')
Data types after F3IwRkvDrGTf:
provinsi           object
tahun              object
jumlah_kasus       object
akses_air_minum    object
dtype: object


In [147]:
from sqlalchemy import text

engine = connect_aiven()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS dbd_airminum"))

In [155]:
def create_table(engine):

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS dbd_airminum (
                id INT PRIMARY KEY,
                provinsi VARCHAR(100),
                tahun INT,
                jumlah_kasus INT,
                akses_air_minum FLOAT
            )
        """))

    print("Tabel berhasil dibuat")

In [149]:
def load_data(engine, df):

    df.to_sql(
        "dbd_airminum",
        engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi"
    )

    print("Data berhasil dikirim ke Aiven")

In [150]:
import pandas as pd

def check_data(engine):

    query = """
    SELECT *
    FROM dbd_airminum
    LIMIT 10
    """

    return pd.read_sql(query, engine)

In [151]:
print(data_final.columns)
print(data_final.dtypes)
print(data_final.columns.tolist())

Index(['id'], dtype='object')
id    int64
dtype: object
['id']


In [152]:
# ==================================================
# 14. DATA VERIFICATION
# Memastikan data berhasil masuk ke Aiven
# ==================================================

engine = connect_aiven()

create_table(engine)

load_data(engine, data_final)

print(check_data(engine))

Tabel berhasil dibuat
Data berhasil dikirim ke Aiven
   id provinsi tahun jumlah_kasus akses_air_minum
0   1     None  None         None            None
1   2     None  None         None            None
2   3     None  None         None            None
3   4     None  None         None            None
4   5     None  None         None            None
5   6     None  None         None            None
6   7     None  None         None            None
7   8     None  None         None            None
8   9     None  None         None            None
9  10     None  None         None            None
